# Task A -- full-data demojized MuRIL plus TF-IDF ensemble

This is the final-fit version of the Task A ensemble. Both component models are
trained once on all 6,401 deduplicated labelled rows, then used to predict the
official Task A validation inputs. No classifier holdout or OOF training is performed
in this notebook.

The blend weights are fixed from the earlier five-fold experiment: 57% demojized
TF-IDF/SVM and 43% demojized MuRIL. They are not refit on the hidden validation
inputs. The purpose is to let both component models learn from the complete labelled
corpus while preserving the previously selected blend.

Expected runtime is approximately 40--60 minutes on a T4. Upload this notebook to
Kaggle, enable GPU and Internet, and choose Save Version -> Save & Run All.

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import numpy as np
import pandas as pd
import torch
from hastika.common.preprocessing import dedupe_index

assert torch.cuda.is_available(), "no GPU -- select a CUDA-enabled runtime"
print("gpu:", torch.cuda.get_device_name(0))
train = pd.read_csv("data/raw/binary_train.csv")
keep = dedupe_index(train["Comment"].tolist(), train["Label"].tolist(), "task A")
print(f"raw labelled rows: {len(train)}; deduplicated rows used for fitting: {len(keep)}")
assert len(keep) == 6401, len(keep)

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Train both components on all labelled data

The SVM `--full-fit` flag skips its OOF pass. MuRIL `--folds 1` means a true full-data
fit; it does not create a 15% holdout. Both commands write probabilities for the
official validation inputs.

In [ ]:
SVM_TAG = "task_a_svm_demojized_full"
MURIL_TAG = "task_a_muril_full"

run([sys.executable, "-u", "-m", "hastika.models.baseline_svm",
     "--task", "a", "--tag", SVM_TAG, "--demojize", "--full-fit"],
    log="artifacts/logs/task_a_svm_demojized_full.log")

run([sys.executable, "-u", "-m", "hastika.models.muril",
     "--tag", MURIL_TAG, "--folds", "1", "--seeds", "42",
     "--epochs", "6"],
    log="artifacts/logs/task_a_muril_full.log")

svm_dir = pathlib.Path("artifacts/runs") / SVM_TAG
muril_dir = pathlib.Path("artifacts/runs") / MURIL_TAG
assert (svm_dir / "test_probs.npy").exists()
assert (muril_dir / "test_probs.npy").exists()
assert not (svm_dir / "oof_probs.npy").exists(), "SVM unexpectedly ran OOF"
assert not (muril_dir / "oof_probs.npy").exists(), "MuRIL unexpectedly ran OOF"
print("both components completed full-data fitting")


## 2. Apply the fixed blend

These weights came from the previous OOF experiment and are deliberately not
re-optimized here. The hidden validation labels must not influence this final fit.

In [ ]:
SVM_WEIGHT = 0.57
MURIL_WEIGHT = 0.43
svm_probs = np.load(svm_dir / "test_probs.npy")
muril_probs = np.load(muril_dir / "test_probs.npy")
assert svm_probs.shape == muril_probs.shape == (806, 2), (svm_probs.shape, muril_probs.shape)
blend_probs = SVM_WEIGHT * svm_probs + MURIL_WEIGHT * muril_probs

blend_tag = "task_a_full_ensemble"
blend_dir = pathlib.Path("artifacts/runs") / blend_tag
blend_dir.mkdir(parents=True, exist_ok=True)
np.save(blend_dir / "test_probs.npy", blend_probs)
validation = pd.read_csv("data/raw/binary_validation_inputs.csv")
labels = np.where(blend_probs[:, 1] > 0.5, "Hate", "Non-Hate")
pd.DataFrame({"id": validation["id"], "label": labels}).to_csv(
    blend_dir / "predictions.csv", index=False)
with open(blend_dir / "blend_weights.json", "w") as f:
    json.dump({"tfidf_svm": SVM_WEIGHT, "muril": MURIL_WEIGHT,
              "training": "full-data"}, f, indent=2)
print("blend weights:", SVM_WEIGHT, MURIL_WEIGHT)
print("prediction distribution:", pd.Series(labels).value_counts().to_dict())


## 3. Package the ensemble submission

The output is a single ZIP containing a bare `predictions.csv` with the required
`id,label` schema. Upload `task_a_full_ensemble.zip` to the Task A validation phase.

In [ ]:
ZIP = pathlib.Path("/kaggle/working/task_a_full_ensemble.zip")
run([sys.executable, "-m", "hastika.common.submission",
     "--task", "a", "--pred", str(blend_dir / "predictions.csv"),
     "--out", str(ZIP)])
with zipfile.ZipFile(ZIP) as z:
    assert z.namelist() == ["predictions.csv"], z.namelist()
print("READY TO UPLOAD:", ZIP)


## 4. Preserve downloadable outputs

In [ ]:
OUT = pathlib.Path("/kaggle/working/task_a_full_ensemble_outputs")
OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(ZIP, OUT / ZIP.name)
shutil.copy2(blend_dir / "predictions.csv", OUT / "predictions.csv")
shutil.copy2(blend_dir / "test_probs.npy", OUT / "test_probs.npy")
shutil.copy2(blend_dir / "blend_weights.json", OUT / "blend_weights.json")
for name in ["task_a_svm_demojized_full.log", "task_a_muril_full.log"]:
    path = pathlib.Path("artifacts/logs") / name
    if path.exists():
        shutil.copy2(path, OUT / name)
print("download directory:", OUT)
